*Importing the needed libraries:*

In [ ]:
import os
import math
import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from typing import List, Tuple, Dict, Optional
from dataclasses import dataclass
from tqdm.notebook import tqdm

from sksurv.util import Surv
from sksurv.metrics import concordance_index_censored
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.impute import SimpleImputer

from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.plotting import add_at_risk_counts
from statsmodels.stats.multitest import multipletests
from lifelines.statistics import logrank_test, multivariate_logrank_test

import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.table import Table
import matplotlib.transforms as mtransforms

import scipy.stats
from scipy.stats import norm, zscore, fisher_exact, chi2_contingency, mannwhitneyu

*Setting the parameters for AI-friendly export format of the plots:*

In [ ]:
import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42   # embed TrueType fonts, keep text editable
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['DejaVu Sans'] 

*Reading the main dataframe with clinical metadata and flow cytometry parameters:*

In [ ]:
# upper day limit for horizoning the clinical follow-up
cutoff = 2000

print('reading the main dataframe:')
# path to the full dataset excel with clinical metadata and flow cytometry parameters for the patients:
full_dataset_clinmetadata_flowcytometrydata_path = ''
d35_input_full = pd.read_excel(full_dataset_clinmetadata_flowcytometrydata_path)
d35_input_full_copy = d35_input_full[(d35_input_full['OC_d35_PassCrit_FCAnalysis'] == 'Yes') | (d35_input_full['VC_d35_PassCrit_FCAnalysis'] == 'Yes')]

PERCENT_FEATURES = [c for c in d35_input_full_copy.columns if ' of ' in c]
myeloid_endings = [' of DC/HLA-DRpos_APC',' of Monocytes',' of Myeloid']
myeloid_endings_and_CD45pos = [' of DC/HLA-DRpos_APC',' of Monocytes',' of Myeloid', ' of CD45pos']
myeloid_child = set(element.split(' of ')[0] for element in PERCENT_FEATURES if element.endswith(tuple(myeloid_endings)))
myeloid_params_exclude = [element for element in PERCENT_FEATURES if element.startswith(tuple(myeloid_child)) and element.endswith(tuple(myeloid_endings_and_CD45pos))]
PERCENT_FEATURES = list(set(PERCENT_FEATURES) - set(myeloid_params_exclude))

PERCENT_FEATURES_renamer = {}
to_rename = [element for element in PERCENT_FEATURES if 'LowLevel' in element]
for element in to_rename:
    PERCENT_FEATURES_renamer[element] = element.replace('LowLevel ', 'High Resolution ')
d35_input_full_copy = d35_input_full_copy.rename(columns = PERCENT_FEATURES_renamer)
PERCENT_FEATURES = [element.replace('LowLevel ', 'High Resolution ') for element in PERCENT_FEATURES]

RATIO_FEATURES   = [c for c in d35_input_full_copy.columns if '-to-' in c]
OUTCOME_COLS = {
    "RFS": {"time": 'Time_to_Relapse_from_TPL', "event": 'Relapse_Present_1_NotObserved_0'},
    "OS":  {"time": 'Time_to_Death_from_TPL',  "event": 'Death_Present_1_NotObserved_0'},
    "aGVHD": {"time": "Time_to_aGVHD", "event": "aGVHD_grading_by_Katja"},
    "cGVHD": {"time": "Time_to_cGVHD", "event": "cGVHD_grading_by_Katja"},
}

d35_input_full_copy['FC_Sampling_Days_After_Allo_d35'] = round(d35_input_full_copy['FC_Sampling_Days_After_Allo_d35'], 0)
d35_input_full_copy['FC_Sampling_Days_After_Allo_d35'] = d35_input_full_copy['FC_Sampling_Days_After_Allo_d35'].astype("Int64")
d35_input_full_copy['Cutoff_Plus_SamplingDay'] = cutoff + d35_input_full_copy['FC_Sampling_Days_After_Allo_d35']

# version with horizoning by a sum of cutoff + sampling day & then landmarking (X days post sampling)
d35_input_full_copy[OUTCOME_COLS['RFS']['event']] = np.where(d35_input_full_copy[OUTCOME_COLS['RFS']['time']] >= d35_input_full_copy['Cutoff_Plus_SamplingDay'], 0, d35_input_full_copy[OUTCOME_COLS['RFS']['event']])
d35_input_full_copy[OUTCOME_COLS['RFS']['time']] = np.where(d35_input_full_copy[OUTCOME_COLS['RFS']['time']] >= d35_input_full_copy['Cutoff_Plus_SamplingDay'], d35_input_full_copy['Cutoff_Plus_SamplingDay'], d35_input_full_copy[OUTCOME_COLS['RFS']['time']])
d35_input_full_copy[OUTCOME_COLS['OS']['event']] = np.where(d35_input_full_copy[OUTCOME_COLS['OS']['time']] >= d35_input_full_copy['Cutoff_Plus_SamplingDay'], 0, d35_input_full_copy[OUTCOME_COLS['OS']['event']])
d35_input_full_copy[OUTCOME_COLS['OS']['time']] = np.where(d35_input_full_copy[OUTCOME_COLS['OS']['time']] >= d35_input_full_copy['Cutoff_Plus_SamplingDay'], d35_input_full_copy['Cutoff_Plus_SamplingDay'], d35_input_full_copy[OUTCOME_COLS['OS']['time']])
d35_input_full_copy[OUTCOME_COLS['aGVHD']['event']] = np.where(d35_input_full_copy[OUTCOME_COLS['aGVHD']['time']] >= d35_input_full_copy['Cutoff_Plus_SamplingDay'], 0, d35_input_full_copy[OUTCOME_COLS['aGVHD']['event']])
d35_input_full_copy[OUTCOME_COLS['aGVHD']['time']] = np.where(d35_input_full_copy[OUTCOME_COLS['aGVHD']['time']] >= d35_input_full_copy['Cutoff_Plus_SamplingDay'], d35_input_full_copy['Cutoff_Plus_SamplingDay'], d35_input_full_copy[OUTCOME_COLS['aGVHD']['time']])
d35_input_full_copy[OUTCOME_COLS['cGVHD']['event']] = np.where(d35_input_full_copy[OUTCOME_COLS['cGVHD']['time']] >= d35_input_full_copy['Cutoff_Plus_SamplingDay'], 0, d35_input_full_copy[OUTCOME_COLS['cGVHD']['event']])
d35_input_full_copy[OUTCOME_COLS['cGVHD']['time']] = np.where(d35_input_full_copy[OUTCOME_COLS['cGVHD']['time']] >= d35_input_full_copy['Cutoff_Plus_SamplingDay'], d35_input_full_copy['Cutoff_Plus_SamplingDay'], d35_input_full_copy[OUTCOME_COLS['cGVHD']['time']])

d35_input_full_copy['Relapse_Present_1_NotObserved_0'] = d35_input_full_copy['Relapse_Present_1_NotObserved_0'].astype("Int64")
d35_input_full_copy['aGVHD_grading_by_Katja'] = d35_input_full_copy['aGVHD_grading_by_Katja'].astype("Int64")
d35_input_full_copy['cGVHD_grading_by_Katja'] = d35_input_full_copy['cGVHD_grading_by_Katja'].astype("Int64")
d35_input_full_copy['MAX_FOLLOW_UP'] = d35_input_full_copy[['Time_to_aGVHD', 'Time_to_cGVHD',
                                                            'Time_to_Relapse_from_TPL','Time_to_Death_from_TPL']].max(axis = 1)


d35_input_full_copy['Time_to_Death_from_TPL'] -= d35_input_full_copy['FC_Sampling_Days_After_Allo_d35']
d35_input_full_copy['Time_to_Relapse_from_TPL'] -= d35_input_full_copy['FC_Sampling_Days_After_Allo_d35']
d35_input_full_copy['Time_to_aGVHD'] -= d35_input_full_copy['FC_Sampling_Days_After_Allo_d35']
# only 65 patients (40 in VC and 25 in OC)
d35_input_full_copy['Time_to_cGVHD'] -= d35_input_full_copy['FC_Sampling_Days_After_Allo_d35']
d35_input_full_copy['MAX_FOLLOW_UP'] -= d35_input_full_copy['FC_Sampling_Days_After_Allo_d35']

*Reshaping the metadata columns for multivariate analysis:*

In [ ]:
df_multivariate = d35_input_full_copy.copy()

df_multivariate['Donor_Relationship_to_Recipient_BrotherSister'] = np.where(df_multivariate['Donor_Relationship_to_Recipient'].isin(['Bruder', 'Schwester']), 'Brother_Sister', 'Other')
df_multivariate['Donor_Relationship_to_Recipient_simplified'] = np.where(df_multivariate['Donor_Relationship_to_Recipient'].isin(['Bruder', 'Sohn', 'Schwester', 'Mutter', 'Cousine','Tochter']), 'Related', 'Unrelated')

df_multivariate['HLA_Category_Final'] = np.where(
    (df_multivariate['Donor_Relationship_to_Recipient_BrotherSister'] == 'Brother_Sister') & 
    (df_multivariate['HLA_Category_refined'].isin(['10/10'])),
    'MSD (10/10)', 'Other')
df_multivariate['HLA_Category_Final'] = np.where(
    (df_multivariate['Donor_Relationship_to_Recipient_simplified'] == 'Related') & 
    (df_multivariate['HLA_Category_refined'].isin(['08/10','09/10'])),
    'MMRD', df_multivariate['HLA_Category_Final'])
df_multivariate['HLA_Category_Final'] = np.where(
    (df_multivariate['Donor_Relationship_to_Recipient_simplified'] == 'Unrelated') & 
    (df_multivariate['HLA_Category_refined'].isin(['10/10'])),
    'MUD (10/10)', df_multivariate['HLA_Category_Final'])
df_multivariate['HLA_Category_Final'] = np.where(
    (df_multivariate['Donor_Relationship_to_Recipient_simplified'] == 'Unrelated') & 
    (df_multivariate['HLA_Category_refined'].isin(['08/10','09/10'])),
    'MMUD', df_multivariate['HLA_Category_Final'])
df_multivariate['HLA_Category_Final'] = np.where(
    df_multivariate['HLA_Category_refined'].isin(['05/10','06/10','07/10']),
    'Haplo', df_multivariate['HLA_Category_Final'])


age_cutoff = 55
df_multivariate[f'Age(>{age_cutoff})'] = np.where(df_multivariate['Age'] > age_cutoff, 1, 0)
age_cutoff = 60
df_multivariate[f'Age(>{age_cutoff})'] = np.where(df_multivariate['Age'] > age_cutoff, 1, 0)
df_multivariate['Age_10y'] = pd.cut(df_multivariate['Age'],
                                    bins=range(0, 101, 10),
                                    labels=[i for i in range(0, 100, 10)],
                                    include_lowest=True)

df_multivariate['MHC_Mismatch_D/R(MM=1)'] = np.where(df_multivariate['MHC_Mismatch_between_Donor&Recipient'] == 'nein', 0, 1)
df_multivariate['ELN_Score(3)'] = np.where(df_multivariate['ELN_Score'] == 3, 1, 0)
df_multivariate['Sex_Recipient(M=1)'] = np.where(df_multivariate['Sex_Recipient'] == 'Male', 1, 0) # Male = 0, Female = 1

df_multivariate['Disease_Score(2)'] = np.where(df_multivariate['Disease_Score'] == 2, 1, 0)
df_multivariate['CMV_Status_R(+)'] = np.where(df_multivariate['CMV_Status_Recipient'] == 'neg', 0, 1)
df_multivariate['CMV_Status_D/R(+/+)'] = np.where((df_multivariate['CMV_Status_Recipient'] == 'pos') & (df_multivariate['CMV_Status_Donor'] == 'pos'), 1, 0)
df_multivariate['EBMT conditioning\n(NMA 0, RIC 1, MAC 2)'] = df_multivariate['EBMT_NMA0_RIC1_MAC2']
df_multivariate['Monosomal KT\nComplex KT\nChr5Δ'] = np.where((d35_input_full_copy['complexKT_1yes'] == 1) | (d35_input_full_copy['monosomal_KT_1yes'] == 1) | (d35_input_full_copy['Monosomy5_Del5'] == 1),
                                                       1, 0)



*Specifying the lists of metadata columns for each kind of multivariate analysis:*

In [ ]:
age_cutoff = 60
clin_var_setup_dict = {
    f'Age55': [f'Age(>55)'],
    f'Age60': [f'Age(>60)'],
    'Age_10y': ['Age_10y'],
    'SexRec': ['Sex_Recipient(M=1)'],
    'MHCMM': ['MHC_Mismatch_D/R(MM=1)'],
    'ELNScore3': ['ELN_Score(3)'],
    'DiseaseScore2': ['Disease_Score(2)'],
    'CMVRec': ['CMV_Status_R(+)'],
    'CMVRecDon': ['CMV_Status_D/R(+/+)'],
    'EBMTRICMAC': ['EBMT_NMA0_RIC1_MAC2'],
    
    f'NoClinVars': ['GvLhi_GvHDlo'],
    f'Age{age_cutoff}_SexRec_MHCMM_ELNScore3': ['GvLhi_GvHDlo', f'Age(>{age_cutoff})','Sex_Recipient(M=1)','MHC_Mismatch_D/R(MM=1)','ELN_Score(3)'],
    f'Age{age_cutoff}_SexRec_MHCMM_ELNScore3_CMVRec': ['GvLhi_GvHDlo', f'Age(>{age_cutoff})','Sex_Recipient(M=1)','CMV_Status_R(+)','MHC_Mismatch_D/R(MM=1)','ELN_Score(3)'],
    f'Age{age_cutoff}_SexRec_MHCMM_ELNScore3_CMVRec_EBMTRICMAC': ['GvLhi_GvHDlo', f'Age(>{age_cutoff})','Sex_Recipient(M=1)','CMV_Status_R(+)','MHC_Mismatch_D/R(MM=1)','ELN_Score(3)','EBMT_NMA0_RIC1_MAC2'],
    f'Age{age_cutoff}_SexRec_MHCMM_ELNScore2022(adv)_CMVRec_EBMTRICMAC': ['GvLhi_GvHDlo', 'MHC_Mismatch_D/R(MM=1)', 'EBMT conditioning\n(NMA 0, RIC 1, MAC 2)','CMV_Status_R(+)','Sex_Recipient(M=1)',f'Age(>{age_cutoff})','ELN_Score(3)'],
    f'Age{age_cutoff}_SexRec_MHCMM_AdvGenetics_CMVRec_EBMTRICMAC': ['GvLhi_GvHDlo', 'MHC_Mismatch_D/R(MM=1)', 'EBMT conditioning\n(NMA 0, RIC 1, MAC 2)','CMV_Status_R(+)','Sex_Recipient(M=1)',f'Age(>{age_cutoff})','Monosomal KT\nComplex KT\nChr5Δ'],
    f'Age{age_cutoff}_SexRec_MHCMM_MonosomalKT_CMVRec_EBMTRICMAC': ['GvLhi_GvHDlo', 'MHC_Mismatch_D/R(MM=1)', 'EBMT conditioning\n(NMA 0, RIC 1, MAC 2)','CMV_Status_R(+)','Sex_Recipient(M=1)',f'Age(>{age_cutoff})','monosomal_KT_1yes'],
    f'Age{age_cutoff}_SexRec_MHCMM_ELNScore2022(adv)_MonosomalKT_CMVRec_EBMTRICMAC': ['GvLhi_GvHDlo', 'MHC_Mismatch_D/R(MM=1)', 'EBMT conditioning\n(NMA 0, RIC 1, MAC 2)','CMV_Status_R(+)','Sex_Recipient(M=1)',f'Age(>{age_cutoff})','monosomal_KT_1yes', 'ELN_Score(3)'],
    f'MonosomalKT': ['monosomal_KT_1yes'],
    f'Age{age_cutoff}_SexRec_MHCMM_ELNScore3_DiseaseScore2_CMVRec_EBMTRICMAC': ['GvLhi_GvHDlo', f'Age(>{age_cutoff})','Sex_Recipient(M=1)','CMV_Status_R(+)','MHC_Mismatch_D/R(MM=1)','ELN_Score(3)','Disease_Score(2)','EBMT_NMA0_RIC1_MAC2'],

    f'NoScore_Age{age_cutoff}_SexRec_MHCMM_ELNScore3': [f'Age(>{age_cutoff})','Sex_Recipient(M=1)','MHC_Mismatch_D/R(MM=1)','ELN_Score(3)'],
    f'NoScore_Age{age_cutoff}_MHCMM_ELNScore3_DiseaseScore': [f'Age(>{age_cutoff})','MHC_Mismatch_D/R(MM=1)','ELN_Score(3)','Disease_Score(2)'],
    f'NoScore_Age{age_cutoff}_SexRec_MHCMM_ELNScore3_DiseaseScore2_CMVRec': [f'Age(>{age_cutoff})','Sex_Recipient(M=1)','CMV_Status_R(+)','MHC_Mismatch_D/R(MM=1)','ELN_Score(3)','Disease_Score(2)'],
    f'NoScore_Age{age_cutoff}_SexRec_MHCMM_ELNScore3_CMVRec': [f'Age(>{age_cutoff})','Sex_Recipient(M=1)','CMV_Status_R(+)','MHC_Mismatch_D/R(MM=1)','ELN_Score(3)'],
    f'NoScore_Age{age_cutoff}_SexRec_MHCMM_ELNScore3_CMVRec_EBMTRICMAC': [f'Age(>{age_cutoff})','Sex_Recipient(M=1)','CMV_Status_R(+)','MHC_Mismatch_D/R(MM=1)','ELN_Score(3)','EBMT_NMA0_RIC1_MAC2'],
    f'NoScore_Age{age_cutoff}_SexRec_MHCMM_ELNScore3_DiseaseScore2_CMVRec_EBMTRICMAC': [f'Age(>{age_cutoff})','Sex_Recipient(M=1)','CMV_Status_R(+)','MHC_Mismatch_D/R(MM=1)','ELN_Score(3)','Disease_Score(2)','EBMT_NMA0_RIC1_MAC2'],
}

*Calculating the ICC35 score (specifying parameters, their direction - for the main ICC35 version and alternatives):*

In [ ]:
score_name = 'CD8TEMGeneral_CD4TCM_CD4TCMNVratio_Score'
# 'CD8TEMGeneral_CD4TCM_CD4TCMNVratio_Score'
# 'CD8TEMGeneral_CD4TCM_CD56dimNK_Score'
# 'CD8TEMGeneral_CD4TCMNVratio_HighResCD56dimNK_Score'
# 'CD8TEMGeneral_CD4TCM_HighResCD56dimNK_Score'
# 'CD8TEMGeneral_CD4TCMNVratio_CD56dimNK_Score'

if score_name == 'CD8TEMGeneral_CD4TCM_CD4TCMNVratio_Score':
    parameters = {
        'CD8 TEM adv1 GPR56+ CD57+ of CD8 TEM': 'GvL_Score',
        'CD4 CCR7mid CD45RA- of T': 'GvHD_Score',
        'CD4_Naive+CD4_TCM-to-CD4_rest': 'GvHD_Score'
    }

    direction = {
        'CD8 TEM adv1 GPR56+ CD57+ of CD8 TEM': 'protective',
        'CD4 CCR7mid CD45RA- of T': 'non-protective',
        'CD4_Naive+CD4_TCM-to-CD4_rest': 'non-protective'
    }

elif score_name == 'CD8TEMGeneral_CD4TCM_CD56dimNK_Score':
    parameters = {
        'CD8 TEM adv1 GPR56+ CD57+ of CD8 TEM': 'GvL_Score',
        "CD56dim NK CD57+ GPR56+ of Lymphocytes": 'GvHD_Score',
        'CD4 CCR7mid CD45RA- of T': 'GvHD_Score',
    }

    direction = {
        'CD8 TEM adv1 GPR56+ CD57+ of CD8 TEM': 'protective',
        "CD56dim NK CD57+ GPR56+ of Lymphocytes": 'protective',
        'CD4 CCR7mid CD45RA- of T': 'non-protective',
    }

elif score_name == 'CD8TEMGeneral_CD4TCM_HighResCD56dimNK_Score':
    parameters = {
        'CD8 TEM adv1 GPR56+ CD57+ of CD8 TEM': 'GvL_Score',
        'High Resolution CD56dim NK CD45RA+ CD57+ GPR56+ of Lymphocytes': 'GvHD_Score',
        'CD4 CCR7mid CD45RA- of T': 'GvHD_Score',
    }

    direction = {
        'CD8 TEM adv1 GPR56+ CD57+ of CD8 TEM': 'protective',
        'High Resolution CD56dim NK CD45RA+ CD57+ GPR56+ of Lymphocytes': 'protective',
        'CD4 CCR7mid CD45RA- of T': 'non-protective',
    }

elif score_name == 'CD8TEMGeneral_CD4TCMNVratio_CD56dimNK_Score':
    parameters = {
        'CD8 TEM adv1 GPR56+ CD57+ of CD8 TEM': 'GvL_Score',
        "CD56dim NK CD57+ GPR56+ of Lymphocytes": 'GvHD_Score',
        'CD4_Naive+CD4_TCM-to-CD4_rest': 'GvHD_Score'
    }

    direction = {
        'CD8 TEM adv1 GPR56+ CD57+ of CD8 TEM': 'protective',
        "CD56dim NK CD57+ GPR56+ of Lymphocytes": 'protective',
        'CD4_Naive+CD4_TCM-to-CD4_rest': 'non-protective'
    }

elif score_name == 'CD8TEMGeneral_CD4TCMNVratio_HighResCD56dimNK_Score':
    parameters = {
        'CD8 TEM adv1 GPR56+ CD57+ of CD8 TEM': 'GvL_Score',
        'High Resolution CD56dim NK CD45RA+ CD57+ GPR56+ of Lymphocytes': 'GvHD_Score',
        'CD4_Naive+CD4_TCM-to-CD4_rest': 'GvHD_Score'
    }

    direction = {
        'CD8 TEM adv1 GPR56+ CD57+ of CD8 TEM': 'protective',
        'High Resolution CD56dim NK CD45RA+ CD57+ GPR56+ of Lymphocytes': 'protective',
        'CD4_Naive+CD4_TCM-to-CD4_rest': 'non-protective'
    }

def sign_intepreter(parameter):
    if direction[parameter] == 'protective':
        return [1,0]
    else:
        return [0,1]

cutoffs = {}
for parameter in list(parameters.keys()):
    cutoffs[parameter] = {
        'OC': np.median(df_multivariate[df_multivariate['Cohort'] == 'OC'][parameter]),
        'VC': np.median(df_multivariate[df_multivariate['Cohort'] == 'VC'][parameter])
    }

for parameter in cutoffs:
    df_multivariate[f'{parameter}_SCORE'] = np.where(df_multivariate['Cohort'] == 'OC',
                                                     np.where(df_multivariate[parameter] > cutoffs[parameter]['OC'], sign_intepreter(parameter)[0], sign_intepreter(parameter)[1]),
                                                     np.where(df_multivariate[parameter] > cutoffs[parameter]['VC'], sign_intepreter(parameter)[0], sign_intepreter(parameter)[1])
                                                    )
GvL_params = list({k:v for k,v in parameters.items() if v == 'GvL_Score'}.keys())
GvHD_params = list({k:v for k,v in parameters.items() if v == 'GvHD_Score'}.keys())
df_multivariate['GvL_Score'] = df_multivariate[[f'{element}_SCORE' for element in GvL_params]].sum(axis = 1)
df_multivariate['GvHD_Score'] = df_multivariate[[f'{element}_SCORE' for element in GvHD_params]].sum(axis = 1)
df_multivariate['GvLhi_GvHDlo'] = np.where((df_multivariate['GvL_Score'] == len({k:v for k,v in parameters.items() if v == 'GvL_Score'})) & 
                                         (df_multivariate['GvHD_Score'] == len({k:v for k,v in parameters.items() if v == 'GvHD_Score'})), 1, 0)

df_multivariate_OC = df_multivariate[df_multivariate['Cohort'] == 'OC']
df_multivariate_VC = df_multivariate[df_multivariate['Cohort'] == 'VC']

*Running the multivariate Cox proportional hazard models:*

In [ ]:
# selecting the penalizer value for multivariate analysis
penalizer_selected = 0.01

def fmt_p(p):
    # return "p<0.001" if p < 0.001 else f"p={p:.3f}"
    return "<0.001" if p < 0.001 else f"{p:.3f}"

renamer_dict = {
    'MHC_Mismatch_D/R(MM=1)': 'MHC Mismatch Don/Rec\n(Mismatch = 1)',
    'ELN_Score(3)': 'ELN Score 2022\n(Adv risk (3) = 1)',
    'Sex_Recipient(M=1)': 'Sex Rec\n(Male = 1)',
    'CMV_Status_R(+)': 'CMV Status Rec\n(CMV+ = 1)',
    'Age(>60)': 'Age\n(>60 = 1)',
    'GvLhi_GvHDlo': 'ICC35\n(favorable = 1)',
    'monosomal_KT_1yes': 'Monosomal KT'
}

for clin_var_setup, columns_to_include in clin_var_setup_dict.items():
    selected_clin_var = 'OS'
    d35_cohort_OC_HR = df_multivariate_OC.copy()
    d35_cohort_VC_HR = df_multivariate_VC.copy()

    columns = [
        OUTCOME_COLS[selected_clin_var]['time'],
        OUTCOME_COLS[selected_clin_var]['event'],
    ] + columns_to_include
    if 'GvLhi_GvHDlo' in columns_to_include:
        order = ['GvLhi_GvHDlo'] + [element for element in columns_to_include if element != 'GvLhi_GvHDlo']
    else:
        order = columns_to_include

    d35_cohort_OC_HR = d35_cohort_OC_HR[columns].copy()
    d35_cohort_OC_HR = d35_cohort_OC_HR.dropna(axis = 0)
    d35_cohort_VC_HR = d35_cohort_VC_HR[columns].copy()
    d35_cohort_VC_HR = d35_cohort_VC_HR.dropna(axis = 0)
    print(f'Final patient count Cohort 1: {len(d35_cohort_OC_HR)}')
    print(f'Final patient count Cohort 2: {len(d35_cohort_VC_HR)}')
        
    fig, axes = plt.subplots(1, 2, figsize=(8, 0.5 * len(order)))
    for working_subset, cohort in zip([d35_cohort_OC_HR, d35_cohort_VC_HR], ['Cohort 1', 'Cohort 2']):
        # print(cohort)
        cph = CoxPHFitter(penalizer=penalizer_selected)
        # cph = CoxPHFitter()
        cph.fit(
            working_subset,
            duration_col=OUTCOME_COLS[selected_clin_var]['time'],
            event_col=OUTCOME_COLS[selected_clin_var]['event'],
        )

        summ = cph.summary.copy()
        summ = summ.loc[order]
        hr = summ["exp(coef)"]
        ci_low = summ["exp(coef) lower 95%"]
        ci_high = summ["exp(coef) upper 95%"]
        y = np.arange(len(order))

        if cohort == 'Cohort 1':
            ax_chosen = axes[0]
        else:
            ax_chosen = axes[1]

        ax_chosen.axvline(1, linestyle='--', color='gray')
        ax_chosen.errorbar(
            hr,
            y,
            xerr=[hr - ci_low, ci_high - hr],
            fmt='s',
            capsize=3,
            color = 'black'
        )


        pvals = summ["p"]
        trans = mtransforms.blended_transform_factory(
            ax_chosen.transAxes,   # x = fraction of axis width
            ax_chosen.transData    # y = your row coordinate
        )
        for yi, var in zip(y, order):
            # label = f"HR={hr.loc[var]:.2f} ({fmt_p(pvals.loc[var])})"
            label = f"{hr.loc[var]:.2f} ({fmt_p(pvals.loc[var])})"
            
            ax_chosen.text(
                0.97, yi, label,
                transform=trans,
                ha="right",
                va="center",
                fontsize=10,
                bbox=dict(
                    boxstyle="round,pad=0.2",
                    facecolor="white",
                    edgecolor="black",
                    alpha=1
                )
            )

        ax_chosen.set_xlim(left=None, right=max(ci_high) + 2.5)
        # ax_chosen.set_xlim(left=0, right=max(ci_high) + 4.5)
        ax_chosen.set_yticks(y)
        if cohort == 'Cohort 1':
            new_label = [renamer_dict[element] if element in renamer_dict else element for element in order]
            ax_chosen.set_yticklabels(new_label)
        else:
            ax_chosen.set_yticklabels([])
        # ax.invert_yaxis()
        ax_chosen.set_xlabel("HR (95% CI)")
        ax_chosen.set_title(f'{cohort}', fontsize=10)
        ax_chosen.grid(True, which='both', linestyle='--', linewidth=0.5, alpha=0.7)

    base_dir_save = '/g/scb2/zaugg/stapran/24_09_FACS_Project_Iter2/SCRIPTS/Plots_For_Paper/Figure4/MultivariateCheck/'
    # plt.suptitle(f'Factors influencing overall survival in two cohorts\nCD4 TCM parameter selected: {CD4_TCM_incl}')
    plt.tight_layout()
    if 'GvLhi_GvHDlo' in columns_to_include:
        plt.savefig(f'{base_dir_save}/{score_name}/{score_name}___ClinVar_{clin_var_setup}_HR_PVAL_ADDED_lambda{penalizer_selected}.pdf', bbox_inches = 'tight')
    else:
        plt.savefig(f'{base_dir_save}/ClinVar/ClinVar_{clin_var_setup}_HR_PVAL_ADDED_lambda{penalizer_selected}.pdf', bbox_inches = 'tight')
    plt.show()

*Running the multivariate analysis specifically for Cohort 2 (genetic risk parameters):*

In [ ]:
subset_cohort2 = df_multivariate_VC.copy()
subset_cohort2['ELN_Score(1)'] = np.where(subset_cohort2['ELN_Score'] == 1, 1, 0)
subset_cohort2['ELN_Score(3)'] = np.where(subset_cohort2['ELN_Score'] == 3, 1, 0)
subset_cohort2['MonKT_or_Chr5Aber'] = np.where((subset_cohort2['monosomal_KT_1yes'] == 1) | (subset_cohort2['Monosomy5_Del5'] == 1), 1, 0)
subset_cohort2 = subset_cohort2.rename(columns = {'GvLhi_GvHDlo': 'ICC35'})
subset_cohort2['ELN3 w/o MDS-rel. mut.'] = np.where((subset_cohort2['ELN_Score'] == 3) & (~subset_cohort2['MDS-related molecular_mutations_0_1_2is2ormore'].isin([1,2])),1,0)


columns_to_include = [
    'ICC35',
    'MHC_Mismatch_D/R(MM=1)',
    'EBMT conditioning\n(NMA 0, RIC 1, MAC 2)',
    'CMV_Status_R(+)',
    'Sex_Recipient(M=1)',
    f'Age(>60)',
    # 'monosomal_KT_1yes',
    'ELN3 w/o MDS-rel. mut.',

    # 'ELN_Score',
    # 'ELN_Score(3)',
    # 'ELN_Score(1)',

    # 'TP53mut_or_17p_aberration',
    # 'MDS-related molecular_mutations_0_1_2is2ormore',
    # 'complexKT_1yes',
    # 'monosomal_KT_1yes',
    # 'Trisomy8_1yes',
    # 'Monosomy7_Del7',
    # 'Monosomy5_Del5',
    # 'MLL_rearr11q23',
    # 'FLT3_ITD',
    # 'MECOM',
    
    # 'MonKT_or_Chr5Aber',
]

penalizer_selected = 0.01
figsize_selected = (6, 0.5 * len(columns_to_include))

def fmt_p(p):
    # return "p<0.001" if p < 0.001 else f"p={p:.3f}"
    return "<0.001" if p < 0.001 else f"{p:.3f}"


selected_clin_var = 'OS'
d35_cohort_VC_HR = subset_cohort2.copy()

columns = [
    OUTCOME_COLS[selected_clin_var]['time'],
    OUTCOME_COLS[selected_clin_var]['event'],
] + columns_to_include
if 'ICC35' in columns_to_include:
    order = ['ICC35'] + [element for element in columns_to_include if element != 'ICC35']
else:
    order = columns_to_include

d35_cohort_VC_HR = d35_cohort_VC_HR[columns].copy()
        
fig, ax = plt.subplots(1, 1, figsize=figsize_selected)
working_subset = d35_cohort_VC_HR.copy()
cohort = 'Cohort 2'
cph = CoxPHFitter(penalizer=penalizer_selected)
cph.fit(
    working_subset,
    duration_col=OUTCOME_COLS[selected_clin_var]['time'],
    event_col=OUTCOME_COLS[selected_clin_var]['event'],
)

summ = cph.summary.copy()
summ = summ.loc[order]
hr = summ["exp(coef)"]
ci_low = summ["exp(coef) lower 95%"]
ci_high = summ["exp(coef) upper 95%"]
y = np.arange(len(order))

ax.axvline(1, linestyle='--', color='gray')
ax.errorbar(
    hr,
    y,
    xerr=[hr - ci_low, ci_high - hr],
    fmt='s',
    capsize=3,
    color = 'black'
)


pvals = summ["p"]
trans = mtransforms.blended_transform_factory(
    ax.transAxes,   # x = fraction of axis width
    ax.transData    # y = your row coordinate
)
for yi, var in zip(y, order):
    # label = f"HR={hr.loc[var]:.2f} ({fmt_p(pvals.loc[var])})"
    label = f"{hr.loc[var]:.2f} ({fmt_p(pvals.loc[var])})"
    
    ax.text(
        0.97, yi, label,
        transform=trans,
        ha="right",
        va="center",
        fontsize=10,
        bbox=dict(
            boxstyle="round,pad=0.2",
            facecolor="white",
            edgecolor="black",
            alpha=1
        )
    )

ax.set_xlim(left=None, right=max(ci_high) + 2.5)
# ax_chosen.set_xlim(left=0, right=max(ci_high) + 4.5)
ax.set_yticks(y)
ax.set_yticklabels(order)
# ax.invert_yaxis()
ax.set_xlabel("HR (95% CI)")
ax.set_title(f'{cohort}', fontsize=10)
ax.grid(True, which='both', linestyle='--', linewidth=0.5, alpha=0.7)

base_dir_save = '/g/scb2/zaugg/stapran/24_09_FACS_Project_Iter2/SCRIPTS/Plots_For_Paper/Figure4/MultivariateCheck/'
# plt.suptitle(f'Factors influencing overall survival in two cohorts\nCD4 TCM parameter selected: {CD4_TCM_incl}')
plt.tight_layout()
if 'ICC35' in columns_to_include:
    plt.savefig(f'{base_dir_save}/{score_name}/Cohort2_{score_name}___ClinVar_{clin_var_setup}_HR_PVAL_ADDED_lambda{penalizer_selected}.pdf', bbox_inches = 'tight')
else:
    plt.savefig(f'{base_dir_save}/ClinVar/Cohort2_ClinVar_{clin_var_setup}_HR_PVAL_ADDED_lambda{penalizer_selected}.pdf', bbox_inches = 'tight')
plt.show()


<br></br>
<br></br>
<br></br>
*Testing for association of the score with ClinVars (Fisher test)*

In [ ]:
score_col = "GvLhi_GvHDlo"   # values: "good", "bad"
cohort = 'Cohort 2'

if cohort == 'Cohort 1':
    df = df_multivariate_OC.copy()
elif cohort == 'Cohort 2':
    df = df_multivariate_VC.copy()
    

categorical_vars = [
    # "ELN_Score",
    'ELN_Score(3)',
    'monosomal_KT_1yes',
    'MHC_Mismatch_D/R(MM=1)',
    'CMV_Status_R(+)',
    'Sex_Recipient(M=1)',
    # 'Disease_Score(2)',
    # 'Disease_Score',
    'Age(>60)',
    'EBMT_NMA0_RIC1_MAC2'
]

continuous_vars = [
    # "Age",
]

results = []

# categorical variables
for var in categorical_vars:
    tmp = df[[score_col, var]].dropna()
    table = pd.crosstab(tmp[score_col], tmp[var])

    res = fisher_exact(table)

    if table.shape == (2, 2):
        test = "Fisher exact (OR)"
    else:
        test = "Fisher exact (Fisher statistic)"
    statistic = res.statistic
    pval = res.pvalue


    results.append({
        "Variable": var,
        "Type": "categorical",
        "Test": test,
        "Pvalue": pval,
        "Fisher OR / Statistic": statistic,
        "N": len(tmp)
    })

# continuous variables
for var in continuous_vars:
    tmp = df[[score_col, var]].dropna()

    good = tmp.loc[tmp[score_col] == 1, var]
    bad  = tmp.loc[tmp[score_col] == 0, var]

    stat, pval = mannwhitneyu(good, bad, alternative="two-sided")

    results.append({
        "Variable": var,
        "Type": "continuous",
        "Test": "Mann-Whitney U",
        "Pvalue": pval,
        "Median_good": good.median(),
        "Median_bad": bad.median(),
        "N_good": len(good),
        "N_bad": len(bad)
    })

results_df = pd.DataFrame(results)
results_df["FDR"] = multipletests(results_df["Pvalue"], method="fdr_bh")[1]

results_df = results_df.sort_values("Pvalue")
results_df['Cohort'] = cohort
base_dir = '/g/scb2/zaugg/stapran/24_09_FACS_Project_Iter2/SCRIPTS/Plots_For_Paper/Figure4/MultivariateCheck/ScoreAssociation_WithClinicalVariables/'

results_df = results_df.rename(columns = {
    'Variable': 'Clinical variable',
    'Type': 'Clinical variable type',
    'Test': 'Statistical test used for association check',
    'Pvalue': 'Unadjusted p-value',
    # 'Fisher OR / Statistic': 'Fisher OR / Statistic',
    'N': 'Number of patients',
    'FDR': 'FDR-adjusted p-value'
})
if cohort == 'Cohort 1':
    results_df.to_excel(f'{base_dir}/Score_{score_name}_C1_TestingAssociationWithScore.xlsx', index = False)
if cohort == 'Cohort 2':
    results_df.to_excel(f'{base_dir}/Score_{score_name}_C2_TestingAssociationWithScore.xlsx', index = False)
results_df

In [ ]:
df_multivariate_VC[[
    'Sex_Recipient(M=1)',
    'GvLhi_GvHDlo'
]].value_counts()

*Individual & multivariate HRs:*

In [ ]:
selected_clin_var = 'OS'

categorical_vars = [
    'Age(>60)',
    'CMV_Status_R(+)',
    'EBMT_NMA0_RIC1_MAC2',
    'ELN_Score(3)',
    'MHC_Mismatch_D/R(MM=1)',
    'Sex_Recipient(M=1)',
    'monosomal_KT_1yes'
    
]

HR_values = {
    'Cohort 1': {
        'univariate': {
            'ELN_Score(3)': {},
            'MHC_Mismatch_D/R(MM=1)': {},
            'CMV_Status_R(+)': {},
            'Sex_Recipient(M=1)': {},
            'Age(>60)': {},
            'EBMT_NMA0_RIC1_MAC2': {},
            'monosomal_KT_1yes': {}
        },
        'multivariate': {
            'ELN_Score(3)': {},
            'MHC_Mismatch_D/R(MM=1)': {},
            'CMV_Status_R(+)': {},
            'Sex_Recipient(M=1)': {},
            'Age(>60)': {},
            'EBMT_NMA0_RIC1_MAC2': {},
            'monosomal_KT_1yes': {}
        },
    },
    'Cohort 2': {
        'univariate': {
            'ELN_Score(3)': {},
            'MHC_Mismatch_D/R(MM=1)': {},
            'CMV_Status_R(+)': {},
            'Sex_Recipient(M=1)': {},
            'Age(>60)': {},
            'EBMT_NMA0_RIC1_MAC2': {},
            'monosomal_KT_1yes': {}
        },
        'multivariate': {
            'ELN_Score(3)': {},
            'MHC_Mismatch_D/R(MM=1)': {},
            'CMV_Status_R(+)': {},
            'Sex_Recipient(M=1)': {},
            'Age(>60)': {},
            'EBMT_NMA0_RIC1_MAC2': {},
            'monosomal_KT_1yes': {}
        },
    },
}
d35_cohort_OC_HR = df_multivariate_OC[categorical_vars + [OUTCOME_COLS[selected_clin_var]['time'],OUTCOME_COLS[selected_clin_var]['event']]].copy()
d35_cohort_VC_HR = df_multivariate_VC[categorical_vars + [OUTCOME_COLS[selected_clin_var]['time'],OUTCOME_COLS[selected_clin_var]['event']]].copy()


for cohort, df in zip(['Cohort 1', 'Cohort 2'], [d35_cohort_OC_HR, d35_cohort_VC_HR]):
    df = df.dropna()
    cph = CoxPHFitter()
    cph.fit(
        df,
        duration_col=OUTCOME_COLS[selected_clin_var]['time'],
        event_col=OUTCOME_COLS[selected_clin_var]['event'],
    )
    summ = cph.summary.copy()
    for variable in categorical_vars:
        HR_values[cohort]['multivariate'][variable]['HR'] = summ.loc[variable, "exp(coef)"]
        HR_values[cohort]['multivariate'][variable]['CI(95%)Lower'] = summ.loc[variable, "exp(coef) lower 95%"]
        HR_values[cohort]['multivariate'][variable]['CI(95%)Upper'] = summ.loc[variable, "exp(coef) upper 95%"]
        HR_values[cohort]['multivariate'][variable]['p-value'] = summ.loc[variable, "p"]
        HR_values[cohort]['multivariate'][variable]['z-value'] = summ.loc[variable, "z"]

for cohort, df in zip(['Cohort 1', 'Cohort 2'], [d35_cohort_OC_HR, d35_cohort_VC_HR]):
    for variable in categorical_vars:
        subset = df[[variable] + [OUTCOME_COLS[selected_clin_var]['time'],OUTCOME_COLS[selected_clin_var]['event']]].copy()
        subset = subset.dropna()
        cph = CoxPHFitter()
        cph.fit(
            subset,
            duration_col=OUTCOME_COLS[selected_clin_var]['time'],
            event_col=OUTCOME_COLS[selected_clin_var]['event'],
        )
        summ = cph.summary.copy()
        HR_values[cohort]['univariate'][variable]['HR'] = summ.loc[variable, "exp(coef)"]
        HR_values[cohort]['univariate'][variable]['CI(95%)Lower'] = summ.loc[variable, "exp(coef) lower 95%"]
        HR_values[cohort]['univariate'][variable]['CI(95%)Upper'] = summ.loc[variable, "exp(coef) upper 95%"]
        HR_values[cohort]['univariate'][variable]['p-value'] = summ.loc[variable, "p"]
        HR_values[cohort]['univariate'][variable]['z-value'] = summ.loc[variable, "z"]



In [ ]:
cohort = 'Cohort 2'
main_dict = HR_values[cohort]
merged = {}

for variable in categorical_vars:
    merged[variable] = {}
    subset_dict_univar= {f'{k} univariate':v for k,v in HR_values[cohort]['univariate'][variable].items()}
    for key,value in subset_dict_univar.items():
        merged[variable][key] = value
    subset_dict_multivar= {f'{k} multivariate':v for k,v in HR_values[cohort]['multivariate'][variable].items()}
    for key,value in subset_dict_multivar.items():
        merged[variable][key] = value
output = pd.DataFrame(merged).T
base_dir = '/g/scb2/zaugg/stapran/24_09_FACS_Project_Iter2/SCRIPTS/Plots_For_Paper/Figure4/MultivariateCheck/IndividualClinicalVariables_UniMultivariate/'
output.to_excel(f'{base_dir}{cohort.replace(" ","")}Age(>60)_CMVRec_EBMTRICMAC_ELNScore3_MonosomalKT_MHCMM_SexRec.xlsx')

*Fisher test & Barplot visualisation for AML score VS CMV_Rec status in both cohorts:*

In [ ]:
def add_bar_percent_labels(ax, plot_df, order, hue_order):
    lookup = {
        (row['GvLhi_GvHDlo'], row['CMV_Status_R(+)']): row['percent_within_score']
        for _, row in plot_df.iterrows()
    }

    for container, hue in zip(ax.containers, hue_order):
        for bar, x_label in zip(container, order):
            pct = lookup.get((x_label, hue), np.nan)
            if not np.isnan(pct):
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_height(),
                    f'{pct:.1f}%',
                    ha='center',
                    va='bottom',
                    fontsize=10
                )

GvLhi_GvHDlo_label = 'GoodComp'
rest_label = 'Other'

CMV_GvL_output = pd.DataFrame(df_multivariate[['GvLhi_GvHDlo', 'CMV_Status_R(+)', 'Cohort']].value_counts()).reset_index()
CMV_GvL_output['GvLhi_GvHDlo'] = CMV_GvL_output['GvLhi_GvHDlo'].replace({0: rest_label, 1: GvLhi_GvHDlo_label})
CMV_GvL_output['CMV_Status_R(+)'] = CMV_GvL_output['CMV_Status_R(+)'].replace({0: 'CMV R neg', 1: 'CMV R pos'})
CMV_GvL_output['percent_within_score'] = (
    CMV_GvL_output
    .groupby(['Cohort', 'GvLhi_GvHDlo'])['count']
    .transform(lambda x: 100 * x / x.sum())
)
custom_palette = ["#9FC5E8","#E06666"]

print('OR > 1 → CMV+ enriched in GoodComp')
print('OR < 1 → CMV+ enriched in Other')
print('-----------')
output_dict = {
    'Cohort 1': {
        'OR': 0,
        'p-value': 0,
    },
    'Cohort 2': {
        'OR': 0,
        'p-value': 0,
    }
}

fig, axes = plt.subplots(1, 2, figsize=(8, 3))
# Cohort 1:
tbl = pd.crosstab(
    pd.Categorical(df_multivariate_OC["GvLhi_GvHDlo"], categories=[1, 0]),
    pd.Categorical(df_multivariate_OC['CMV_Status_R(+)'], categories=[1, 0])
)
oddsratio, pvalue = fisher_exact(tbl)
sns.barplot(
    data = CMV_GvL_output[CMV_GvL_output['Cohort'] == 'OC'],
    x = 'GvLhi_GvHDlo',
    y = 'count',
    hue = 'CMV_Status_R(+)',
    ax = axes[0],
    legend = False,
    palette=custom_palette,
    order = [GvLhi_GvHDlo_label, rest_label], hue_order = ['CMV R pos', 'CMV R neg']
)
axes[0].text(0.2, CMV_GvL_output['count'].max()*0.95, f"Fisher's exact\nOR = {oddsratio:.3g}\np = {pvalue:.3g}",
            #  bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray"),
             ha="center",va="center")
axes[0].set_ylim(0, CMV_GvL_output['count'].max() * 1.1)
axes[0].set_ylabel('Patients')
axes[0].set_xlabel('')
print(f'Cohort 1\nOR: {oddsratio:.3g}\np-value: {pvalue:.3g}\n--------')
output_dict['Cohort 1']['OR'], output_dict['Cohort 1']['p-value'] = oddsratio, pvalue
add_bar_percent_labels(
    axes[0],
    CMV_GvL_output[CMV_GvL_output['Cohort'] == 'OC'],
    order=[GvLhi_GvHDlo_label, rest_label],
    hue_order=['CMV R pos', 'CMV R neg']
)



# Cohort 2:
tbl = pd.crosstab(
    pd.Categorical(df_multivariate_VC["GvLhi_GvHDlo"], categories=[1, 0]),
    pd.Categorical(df_multivariate_VC['CMV_Status_R(+)'], categories=[1,0])
)
oddsratio, pvalue = fisher_exact(tbl)
sns.barplot(
    data = CMV_GvL_output[CMV_GvL_output['Cohort'] == 'VC'],
    x = 'GvLhi_GvHDlo',
    y = 'count',
    hue = 'CMV_Status_R(+)',
    ax = axes[1],
    palette=custom_palette,
    order = [GvLhi_GvHDlo_label, rest_label], hue_order = ['CMV R pos', 'CMV R neg']
)

axes[1].set_ylabel('')
axes[1].set_xlabel('')
axes[1].tick_params(axis="y", labelleft=False)
axes[1].text(0.2, CMV_GvL_output['count'].max()*0.95, f"Fisher's exact\nOR = {oddsratio:.3g}\np = {pvalue:.3g}",
            #  bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray"),
             ha="center",va="center")
axes[1].set_ylim(0, CMV_GvL_output['count'].max() * 1.1)

axes[1].legend(bbox_to_anchor=(1,1.01), fontsize = 10,
               title = 'CMV Status (R)') 
print(f'Cohort 2\nOR: {oddsratio:.3g}\np-value: {pvalue:.3g}\n--------')
output_dict['Cohort 2']['OR'],output_dict['Cohort 2']['p-value'] = oddsratio, pvalue
add_bar_percent_labels(
    axes[1],
    CMV_GvL_output[CMV_GvL_output['Cohort'] == 'VC'],
    order=[GvLhi_GvHDlo_label, rest_label],
    hue_order=['CMV R pos', 'CMV R neg']
)



fig.subplots_adjust(wspace=0.05)

save_dir = '/g/scb2/zaugg/stapran/24_09_FACS_Project_Iter2/SCRIPTS/Plots_For_Paper/Figure4/FishersExactTest'
plt.savefig(f'{save_dir}/CMVRec_Score_{score_name}_FisherTest.pdf', bbox_inches = 'tight')
pd.DataFrame(output_dict).to_csv(f'{save_dir}/CMVRec_Score_{score_name}_FisherTest.csv')
plt.show()